# scratch192_lr3e3 — capacity control: FROM SCRATCH 192ch, 40 epochs, UNWEIGHTED r2_bulk (12.51M rows), LR 3e-3. Identical to scratch128_lr3e3 except width. Correct directional D4 is mandatory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v5.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v5.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/r2_bulk.pt.gz /content/r2_bulk.pt.gz
gz=os.path.getsize('/content/r2_bulk.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 740_453_101, f'.gz truncated! got {gz}; re-upload r2_bulk.pt.gz'
!gunzip -t /content/r2_bulk.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/r2_bulk.pt.gz > /content/alphatrain/data/r2_bulk.pt
pt=os.path.getsize('/content/alphatrain/data/r2_bulk.pt')
assert pt == 2_039_621_645, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 12,512,995 rows ({time.time()-t0:.0f}s)')
!rm /content/r2_bulk.pt.gz
!pip install -q numpy numba scipy

# Refuse the historical corrupt D4 implementation.
_dataset_source = open('/content/alphatrain/dataset.py').read()
assert ('def _build_line_direction_luts' in _dataset_source and
        'def _transform_observation' in _dataset_source), (
    'STALE CODE TARBALL: directional H/V/D1/D2 channels are not '
    'transformed under D4; rebuild/upload colorlines_pillar3d_v5.tar.gz.')
print('Exact directional-channel D4 transform confirmed.')


In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (scratch192_lr3e3) =====
CHANNELS = 192
EPOCHS   = 40
BATCH    = 32768
LR       = 3e-3
T        = 1.0
DW       = 0
BLEND    = 0.0
GAMMA    = 0  # unweighted — exact scratch128_lr3e3 objective
SEED     = 42
SAVE_STEPS = 1000
RUN      = "scratch192_lr3e3"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} gamma={GAMMA}')


In [ ]:
%cd /content
train_t0=time.time()
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/r2_bulk.pt \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} --disagree-gamma {GAMMA} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
print(f'TOTAL_TRAIN_WALL_SECONDS={time.time()-train_t0:.1f}')
# Validation CE is informational; distributional gameplay is the gate.


In [ ]:
# Checkpoints save directly to Drive as training proceeds.
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)


## Capacity-test gate

1. Export and screen every third epoch on the aggregate 999-game distribution over `[775000,775999)`. Do not compare individual seed outcomes.
2. Evaluate the apparent peak and any late non-plateau checkpoint on 5,000 games over `[775000,780000)`. Compare mean, median, P5/P10, `<1000`, and long-tail rates with independent bootstrap intervals.
3. Compare against `scratch128_lr3e3` two ways: same epoch/optimizer steps (capacity per datum) and matched wall-clock (system efficiency). Both runs use the same data, seed, batch, LR schedule, objective, and corrected D4.
4. Evidence for a 128ch capacity ceiling requires the tuned 128 trajectory to flatten while 192ch continues to improve broadly, especially on crisis/floor metrics. A higher 192 mean alone is insufficient.
